# Exploratory Analysis of Filtering Sources

Characterizes the expression/abundance data sources used for tissue-specific PPI filtering themselves (distributions, coverage, cross-source agreement) — independent of any particular filtering run or downstream module result. Currently covers GTEx (median TPM); PaxDB is included only via the cross-source tissue-clustering comparison in Section 4.

## 1 — Setup

In [ ]:
from pathlib import Path

import graph_tool.all as gt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path(".").resolve()
NET_DIR   = REPO_ROOT / "alzheimer_symbol_runs/alzheimer_run_symbol_threshold_1/input/networks"
EXPR_FILE = REPO_ROOT / "data/expression_by_tissue.gct"
OUT_ROOT  = REPO_ROOT / "sanity_check_outputs"

TISSUE   = "Brain_Hippocampus"
NETWORKS = {
    "iid":    NET_DIR / "iid.human.Symbol.gt",
    "string": NET_DIR / "string.human_links_v12_0_min900.Symbol.gt",
}

OUT_ROOT.mkdir(exist_ok=True)
print("Expression file:", EXPR_FILE)
print("Networks:", {k: v.name for k, v in NETWORKS.items()})


In [ ]:
def node_names(gt_path):
    g = gt.load_graph(str(gt_path))
    return {g.vp["name"][v] for v in g.iter_vertices()}

# node sets of the raw, unfiltered networks
original_nodes = {name: node_names(path) for name, path in NETWORKS.items()}
print({k: len(v) for k, v in original_nodes.items()})


## 2 — Load GCT expression data (all tissues)

In [ ]:
gct = pd.read_csv(EXPR_FILE, sep="\t", skiprows=2, header=0)
# Description column contains gene symbols; rename for clarity
gct = gct.rename(columns={"Name": "ensembl", "Description": "symbol"})
# strip Ensembl version suffixes
gct["ensembl"] = gct["ensembl"].str.split(".").str[0]

tissue_cols = [c for c in gct.columns if c not in ("ensembl", "symbol")]
print(f"{len(gct):,} genes × {len(tissue_cols)} tissues")
print("Brain tissues:", [t for t in tissue_cols if t.startswith("Brain")])

## 3 — Expression CDF of network nodes, with candidate threshold lines

Where do the network's nodes fall in the GTEx expression distribution for the tissue of interest? Used to sanity-check candidate filtering thresholds before running them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

THRESHOLDS = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

for ax, (net_name, net_path) in zip(axes, NETWORKS.items()):
    nodes = original_nodes[net_name]
    expr_in_net = gct[gct["symbol"].isin(nodes)][TISSUE].dropna().values
    # clip for visibility; log-scale x after
    expr_clipped = np.clip(expr_in_net, 1e-3, None)
    sorted_expr  = np.sort(expr_clipped)
    cdf          = np.arange(1, len(sorted_expr) + 1) / len(sorted_expr)

    ax.plot(sorted_expr, cdf, lw=2, label="CDF")
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(THRESHOLDS)))
    for thr, col in zip(THRESHOLDS, colors):
        ax.axvline(thr, color=col, linestyle="--", lw=1.2, label=f"thr={thr}")
    ax.set_xscale("log")
    ax.set_xlabel("Brain_Hippocampus median TPM (log scale)")
    ax.set_ylabel("Cumulative fraction of network nodes")
    ax.set_title(f"{net_name.upper()} — expression CDF")
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_ROOT / "expression_cdf.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 — Cross-source comparison: GTEx vs. PaxDB tissue clustering

How well do the two expression/abundance sources agree with each other on tissue identity? Computed on the 10 tissues with an unambiguous 1:1 GTEx ↔ PaxDB name mapping, log2-transformed, using both Spearman correlation and cosine similarity as the similarity metric.

In [ ]:
# -----------------------------
# Clustering heatmap of PaxDb + GTEx tissues (Spearman correlation, 1:1 tissue subset)
# -----------------------------
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage

# --- PaxDb (protein abundance, ppm), wide format: id + one column per tissue ---
paxdb = pd.read_csv("paxdb_all_tissues.tsv", sep="\t").set_index("id")
# PaxDb only reports detected proteins per tissue; a missing entry means "not detected" (~0 abundance),
# not "unknown", so fill before joining rather than dropping genes with any missing tissue.
paxdb = paxdb.fillna(0)
paxdb.columns = [f"PaxDb_{c}" for c in paxdb.columns]

# --- GTEx (median gene TPM), wide format: Name (Ensembl), Description (symbol), one column per tissue ---
gtex_raw = pd.read_csv(
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz",
    sep="\t",
    skiprows=2,
)
gtex = gtex_raw.drop(columns=["Name"]).groupby("Description").sum()
gtex.index.name = "id"
gtex.columns = [f"GTEx_{c}" for c in gtex.columns]

# --- combine on shared genes (symbol) and log-transform (abundances span orders of magnitude) ---
combined = paxdb.join(gtex, how="inner")
log_combined = np.log2(combined + 1)

print(f"{combined.shape[0]} shared genes, {paxdb.shape[1]} PaxDb tissues, {gtex.shape[1]} GTEx tissues")

# --- curated subset: 10 tissues with an unambiguous 1:1 mapping between PaxDb and GTEx ---
# (both tables also contain sub-regions/cell types, e.g. GTEx Brain_* or PaxDb SKIN_FIBROBLAST,
# that don't have a single clear counterpart in the other table, so those are excluded here)
PAXDB_TO_GTEX = {
    "ADRENAL_GLAND": "Adrenal_Gland",
    "LIVER": "Liver",
    "LUNG": "Lung",
    "OVARY": "Ovary",
    "PANCREAS": "Pancreas",
    "PROSTATE_GLAND": "Prostate",
    "SPLEEN": "Spleen",
    "TESTIS": "Testis",
    "THYROID_GLAND": "Thyroid",
    "UTERUS": "Uterus",
}

subset_cols = [f"PaxDb_{p}" for p in PAXDB_TO_GTEX] + [f"GTEx_{g}" for g in PAXDB_TO_GTEX.values()]
subset = log_combined[subset_cols]

# --- Spearman rank correlation between tissues (columns) across shared genes, used both for the
#     heatmap color and as the distance metric (1 - correlation) for clustering ---
sim = subset.corr(method="spearman")

dist = 1 - sim.values
np.fill_diagonal(dist, 0)  # avoid tiny negative values from floating-point error
dist = (dist + dist.T) / 2  # enforce exact symmetry for squareform
condensed = squareform(dist, checks=False)
tissue_linkage = linkage(condensed, method="average")

source = pd.Series(
    ["PaxDb" if c.startswith("PaxDb_") else "GTEx" for c in sim.columns],
    index=sim.columns,
)
palette = {"PaxDb": "tab:orange", "GTEx": "tab:blue"}
source_colors = source.map(palette)

g = sns.clustermap(
    sim,
    row_linkage=tissue_linkage,
    col_linkage=tissue_linkage,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    row_colors=source_colors,
    col_colors=source_colors,
    figsize=(12, 12),
    xticklabels=True,
    yticklabels=True,
    dendrogram_ratio=0.15,
    cbar_kws={"label": "Spearman correlation"},
)
g.ax_heatmap.tick_params(labelsize=8)

legend_handles = [Patch(facecolor=color, label=name) for name, color in palette.items()]
g.ax_heatmap.legend(
    handles=legend_handles,
    title="Source",
    loc="upper left",
    bbox_to_anchor=(1.15, 1.0),
    bbox_transform=g.ax_heatmap.transAxes,
    frameon=False,
)

g.savefig("paxdb_gtex_tissue_clustermap.pdf")
plt.show()


In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage
from sklearn.metrics.pairwise import cosine_similarity

paxdb = pd.read_csv("paxdb_all_tissues.tsv", sep="\t").set_index("id")
paxdb = paxdb.fillna(0)
paxdb.columns = [f"PaxDb_{c}" for c in paxdb.columns]

gtex_raw = pd.read_csv(
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz",
    sep="\t",
    skiprows=2,
)
gtex = gtex_raw.drop(columns=["Name"]).groupby("Description").sum()
gtex.index.name = "id"
gtex.columns = [f"GTEx_{c}" for c in gtex.columns]

combined = paxdb.join(gtex, how="inner")
log_combined = np.log2(combined + 1)

print(f"{combined.shape[0]} shared genes, {paxdb.shape[1]} PaxDb tissues, {gtex.shape[1]} GTEx tissues")
print("any NaN in combined:", combined.isna().any().any())

PAXDB_TO_GTEX = {
    "ADRENAL_GLAND": "Adrenal_Gland",
    "LIVER": "Liver",
    "LUNG": "Lung",
    "OVARY": "Ovary",
    "PANCREAS": "Pancreas",
    "PROSTATE_GLAND": "Prostate",
    "SPLEEN": "Spleen",
    "TESTIS": "Testis",
    "THYROID_GLAND": "Thyroid",
    "UTERUS": "Uterus",
}

subset_cols = [f"PaxDb_{p}" for p in PAXDB_TO_GTEX] + [f"GTEx_{g}" for g in PAXDB_TO_GTEX.values()]
subset = log_combined[subset_cols]
print("any NaN in subset:", subset.isna().any().any())

sim = cosine_similarity(subset.T.values)
sim = pd.DataFrame(sim, index=subset_cols, columns=subset_cols)

dist = 1 - sim.values
np.fill_diagonal(dist, 0)
condensed = squareform(dist, checks=False)
tissue_linkage = linkage(condensed, method="average")

source = pd.Series(
    ["PaxDb" if c.startswith("PaxDb_") else "GTEx" for c in sim.columns],
    index=sim.columns,
)
palette = {"PaxDb": "tab:orange", "GTEx": "tab:blue"}
source_colors = source.map(palette)

g = sns.clustermap(
    sim,
    row_linkage=tissue_linkage,
    col_linkage=tissue_linkage,
    cmap="mako",
    vmin=0,
    vmax=1,
    row_colors=source_colors,
    col_colors=source_colors,
    figsize=(12, 12),
    xticklabels=True,
    yticklabels=True,
    dendrogram_ratio=0.15,
    cbar_kws={"label": "cosine similarity"},
)
g.ax_heatmap.tick_params(labelsize=8)

legend_handles = [Patch(facecolor=color, label=name) for name, color in palette.items()]
g.ax_heatmap.legend(
    handles=legend_handles,
    title="Source",
    loc="upper left",
    bbox_to_anchor=(1.15, 1.0),
    bbox_transform=g.ax_heatmap.transAxes,
    frameon=False,
)

g.savefig("paxdb_gtex_tissue_clustermap.pdf")
plt.show()
print("done")


In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage

paxdb = pd.read_csv("paxdb_all_tissues.tsv", sep="\t").set_index("id")
paxdb = paxdb.fillna(0)
paxdb.columns = [f"PaxDb_{c}" for c in paxdb.columns]

gtex_raw = pd.read_csv(
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz",
    sep="\t",
    skiprows=2,
)
gtex = gtex_raw.drop(columns=["Name"]).groupby("Description").sum()
gtex.index.name = "id"
gtex.columns = [f"GTEx_{c}" for c in gtex.columns]

combined = paxdb.join(gtex, how="inner")
log_combined = np.log2(combined + 1)

print(f"{combined.shape[0]} shared genes, {paxdb.shape[1]} PaxDb tissues, {gtex.shape[1]} GTEx tissues")

PAXDB_TO_GTEX = {
    "ADRENAL_GLAND": "Adrenal_Gland",
    "LIVER": "Liver",
    "LUNG": "Lung",
    "OVARY": "Ovary",
    "PANCREAS": "Pancreas",
    "PROSTATE_GLAND": "Prostate",
    "SPLEEN": "Spleen",
    "TESTIS": "Testis",
    "THYROID_GLAND": "Thyroid",
    "UTERUS": "Uterus",
}

subset_cols = [f"PaxDb_{p}" for p in PAXDB_TO_GTEX] + [f"GTEx_{g}" for g in PAXDB_TO_GTEX.values()]
subset = log_combined[subset_cols]

sim = subset.corr(method="spearman")

dist = 1 - sim.values
np.fill_diagonal(dist, 0)
dist = (dist + dist.T) / 2
condensed = squareform(dist, checks=False)
tissue_linkage = linkage(condensed, method="average")

source = pd.Series(
    ["PaxDb" if c.startswith("PaxDb_") else "GTEx" for c in sim.columns],
    index=sim.columns,
)
palette = {"PaxDb": "tab:orange", "GTEx": "tab:blue"}
source_colors = source.map(palette)

g = sns.clustermap(
    sim,
    row_linkage=tissue_linkage,
    col_linkage=tissue_linkage,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    row_colors=source_colors,
    col_colors=source_colors,
    figsize=(12, 12),
    xticklabels=True,
    yticklabels=True,
    dendrogram_ratio=0.15,
    cbar_kws={"label": "Spearman correlation"},
)
g.ax_heatmap.tick_params(labelsize=8)

legend_handles = [Patch(facecolor=color, label=name) for name, color in palette.items()]
g.ax_heatmap.legend(
    handles=legend_handles,
    title="Source",
    loc="upper left",
    bbox_to_anchor=(1.15, 1.0),
    bbox_transform=g.ax_heatmap.transAxes,
    frameon=False,
)

g.savefig("paxdb_gtex_tissue_clustermap.pdf")
plt.show()
print("done")
